# MMAF-CD v25 — Stage F3n: official HoVer-Net (TIAToolbox) on the F nuclei tiles

Runs ONLY the official `hovernet_fast-pannuke` model on the exact tiles Stage F scored our HoVer-Net on
(`pannuke.npz` 150 tiles, `monuseg.npz` 338 tiles), with the same PQ function. ~5–10 min on a T4.

**Inputs (one of):** the Stage F notebook output (contains `g1b/pannuke.npz`, `g1b/monuseg.npz`),
or a dataset you made from those two files. **Accelerator:** GPU. **Internet:** ON (pip + weights).


In [ ]:
import subprocess, sys, json, time
from pathlib import Path
def find(name):
    hits = []
    for root in [Path("/kaggle/input")]:
        for p in root.rglob(name):
            hits.append(p)
    return sorted(hits, key=lambda p: len(p.parts))
D = Path("/kaggle/working/g1b"); D.mkdir(parents=True, exist_ok=True)
for n in ["pannuke.npz", "monuseg.npz"]:
    h = find(n)
    if not h: raise FileNotFoundError(f"{n} not found under /kaggle/input — attach the Stage F output or a dataset with it")
    (D / n).write_bytes(h[0].read_bytes()); print(f"✅ {n} ← {h[0]}  ({h[0].stat().st_size/1e6:.1f} MB)")


In [ ]:
RUNNER = r'''
import sys, json, shutil, numpy as np, cv2, joblib
from pathlib import Path
from PIL import Image
D = Path(sys.argv[1]); out = {}
def _compute_panoptic_quality(inst_pred, inst_gt, iou_threshold=0.5):
    pred_ids   = set(np.unique(inst_pred)) - {0}
    gt_ids     = set(np.unique(inst_gt))   - {0}
    matched_iou = []; matched_pred = set(); matched_gt = set()

    for p_id in pred_ids:
        p_mask   = (inst_pred == p_id)
        best_iou = 0.; best_g = None
        for g_id in gt_ids:
            if g_id in matched_gt: continue
            g_mask = (inst_gt == g_id)
            inter  = float((p_mask & g_mask).sum())
            union  = float((p_mask | g_mask).sum())
            iou    = inter / (union + 1e-6)
            if iou > best_iou:
                best_iou = iou; best_g = g_id
        if best_iou >= iou_threshold:
            matched_iou.append(best_iou)
            matched_pred.add(p_id); matched_gt.add(best_g)

    TP = len(matched_iou); FP = len(pred_ids)-TP; FN = len(gt_ids)-TP
    SQ = float(np.mean(matched_iou)) if matched_iou else 0.
    DQ = TP / (TP + 0.5*FP + 0.5*FN + 1e-6)
    return {"PQ": round(DQ*SQ,4), "DQ": round(DQ,4), "SQ": round(SQ,4),
            "TP": TP, "FP": FP, "FN": FN}

def inst_maps(files, shapes, save):
    # Direct model inference — identical in TIAToolbox 1.x and 2.x (the engines' APIs differ and
    # 2.x needs slide MPP metadata that plain PNG tiles lack). hovernet_fast-pannuke: 256→164 px
    # valid-conv patches at 0.25 mpp → reflect-pad 46 px, tile with stride 164, stitch, then the
    # library's own np/hv watershed post-processing.
    import torch, tiatoolbox
    from tiatoolbox.models.architecture import get_pretrained_model
    try: model, _ = get_pretrained_model("hovernet_fast-pannuke")
    except TypeError: model, _ = get_pretrained_model(pretrained_model="hovernet_fast-pannuke")
    from tiatoolbox.models.architecture.hovernet import HoVerNet as _HV
    model = getattr(model, "module", model)                      # 2 GPUs → TIAToolbox may wrap in DataParallel
    dev = "cuda" if torch.cuda.is_available() else "cpu"; model = model.to(dev).eval()
    proc = _HV._proc_np_hv
    IN, OUT = 256, 164; P = (IN - OUT) // 2
    maps = []
    for f, shp in zip(files, shapes):
        im = np.array(Image.open(f).convert("RGB")); H, W = im.shape[:2]
        pad = np.pad(im, ((P, P + max(0, OUT - H)), (P, P + max(0, OUT - W)), (0, 0)), mode="reflect")
        Hc, Wc = max(H, OUT), max(W, OUT)
        ys = sorted(set(list(range(0, Hc - OUT, OUT)) + [Hc - OUT])); xs = sorted(set(list(range(0, Wc - OUT, OUT)) + [Wc - OUT]))
        win = [(y, x) for y in ys for x in xs]
        bt = torch.from_numpy(np.stack([pad[y:y + IN, x:x + IN] for y, x in win])).permute(0, 3, 1, 2).float().to(dev)
        with torch.inference_mode():
            o = model(bt)
            npb = torch.softmax(o["np"], 1)[:, 1].cpu().numpy(); hvb = o["hv"].permute(0, 2, 3, 1).cpu().numpy()
        npm = np.zeros((Hc, Wc), np.float32); hvm = np.zeros((Hc, Wc, 2), np.float32)
        for (y, x), a_, h_ in zip(win, npb, hvb): npm[y:y + OUT, x:x + OUT] = a_; hvm[y:y + OUT, x:x + OUT] = h_
        inst = np.asarray(proc(npm[:H, :W, None], hvm[:H, :W])).astype(np.int32)
        if inst.shape != tuple(shp): inst = cv2.resize(inst, (shp[1], shp[0]), interpolation=cv2.INTER_NEAREST)
        maps.append(inst)
    return maps, f"direct-model tiatoolbox {tiatoolbox.__version__}"
for npz in sorted(D.glob("*.npz")):
    z = np.load(npz); imgs, gts = z["imgs"], z["gts"]
    tmp = D / f"in_{npz.stem}"; shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
    files = []
    for i, im in enumerate(imgs):
        f = tmp / f"t{i:05d}.png"; Image.fromarray(im).save(f); files.append(str(f))
    save = D / f"out_{npz.stem}"; shutil.rmtree(save, ignore_errors=True)
    try:
        maps, api = inst_maps(files, [g.shape for g in gts], save)
        r = [_compute_panoptic_quality(p, g) for p, g in zip(maps, gts)]
        out[npz.stem] = {k: float(np.mean([x[k] for x in r])) for k in ["PQ", "DQ", "SQ"]}
        out[npz.stem].update(PQ_std=float(np.std([x["PQ"] for x in r])), n=len(r), api=api,
                             pred_nuclei_per_tile=float(np.mean([len(np.unique(p)) - 1 for p in maps])),
                             gt_nuclei_per_tile=float(np.mean([len(np.unique(g)) - 1 for g in gts])))
        print(npz.stem, out[npz.stem], flush=True)
    except Exception as e:
        import traceback; traceback.print_exc(); out[npz.stem] = {"error": f"{type(e).__name__}: {e}"}
(D / "tia_results.json").write_text(json.dumps(out, indent=1))
'''
(D / "tia_runner.py").write_text(RUNNER)
import py_compile; py_compile.compile(str(D / "tia_runner.py"), doraise=True)
print("installing TIAToolbox …", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiatoolbox"], check=False)
t = time.time()
p = subprocess.run([sys.executable, str(D / "tia_runner.py"), str(D)], capture_output=True, text=True, timeout=3600)
print("\n".join(l for l in (p.stdout + p.stderr).splitlines() if not any(s in l for s in ("Warning", "warn(", "━")))[-6000:])
print(f"elapsed {time.time() - t:.0f}s")


In [ ]:
res = json.loads((D / "tia_results.json").read_text())
MMAF = {"pannuke": dict(PQ=0.3259, PQ_std=0.176, DQ=0.4363, SQ=0.6798, n=150, pred=19.1, gt=29.1),
        "monuseg": dict(PQ=0.2805, PQ_std=0.170, DQ=0.4097, SQ=0.6115, n=338, pred=22.1, gt=42.2)}   # Stage F run
print(f"{'set':<9}{'model':<26}{'PQ':>14}{'DQ':>8}{'SQ':>8}{'nuclei/tile':>14}")
for k in ["pannuke", "monuseg"]:
    m = MMAF[k]; print(f"{k:<9}{'MMAF HoVer-Net (ours)':<26}{m['PQ']:>8.4f}±{m['PQ_std']:.3f}{m['DQ']:>8.4f}{m['SQ']:>8.4f}{m['pred']:>8.1f} / {m['gt']:.1f}")
    v = res.get(k, {})
    if "error" in v or not v: print(f"{k:<9}{'official HoVer-Net':<26}  ERROR: {v.get('error', 'missing')}"); continue
    print(f"{k:<9}{'official HoVer-Net':<26}{v['PQ']:>8.4f}±{v['PQ_std']:.3f}{v['DQ']:>8.4f}{v['SQ']:>8.4f}{v['pred_nuclei_per_tile']:>8.1f} / {v['gt_nuclei_per_tile']:.1f}")
json.dump({"mmaf": MMAF, "official": res}, open("/kaggle/working/f3n_nuclei_results.json", "w"), indent=1)
print("\n✅ /kaggle/working/f3n_nuclei_results.json")
